<a href="https://colab.research.google.com/github/sohailpayami2023/digital-signal-processing-python/blob/main/notebooks/00_python_foundations/02_complex_numbers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Complex Numbers and Phasors for DSP Engineers

Complex numbers are the mathematical language of DSP and
wireless communications. Every signal, channel coefficient,
and FFT bin is complex. This notebook builds the foundation
from arithmetic through to phasor diagrams and dB.

**Topics covered**
1. Why complex numbers in DSP?
2. Rectangular form and arithmetic
3. Polar form and Euler's formula
4. NumPy operations on complex arrays
5. Argand (phasor) diagrams
6. Rotating phasors and complex exponentials
7. dB and dBm
8. MATLAB → Python quick reference


# 1. Why Complex Numbers in DSP?

A real sinusoid `cos(2πft)` has both positive and
negative frequency components — the FFT cannot separate
them. The **complex exponential** `e^(j2πft)` has
only ONE frequency component, making analysis cleaner.

In wireless communications:
- **IQ signals** — I (in-phase) + jQ (quadrature) form
  the complex baseband representation of any RF signal
- **Channel coefficients** h ∈ ℂ — magnitude = path loss,
  phase = propagation delay
- **FFT output** — each bin X[k] is complex;
  magnitude = amplitude, angle = phase of that frequency
- **Constellation points** — QPSK, QAM symbols live on
  the complex plane


# 2. Rectangular Form and Arithmetic

A complex number z = a + jb has:
- **Real part** a = Re{z}
- **Imaginary part** b = Im{z}

> Python (and engineering) uses `j` for √−1.
> MATLAB accepts both `i` and `j`.
> Never write `j` as a standalone variable in Python
> before assigning it — `j` is just a name until you
> write a literal like `1j` or `0+1j`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.figsize' : (5, 3),
    'figure.dpi'     : 120,
    'axes.grid'      : True,
    'grid.alpha'     : 0.4,
})

# Literal complex numbers — MATLAB: z = 2 + 3i
z1 = 2 + 3j
z2 = 1 - 1j

# Real and imaginary parts
print('z1       :', z1)
print('Re{z1}   :', z1.real)   # MATLAB: real(z1)
print('Im{z1}   :', z1.imag)   # MATLAB: imag(z1)
print('type     :', type(z1))

# Arithmetic — same operators as real numbers
print('z1 + z2  :', z1 + z2)   # (3+2j)
print('z1 - z2  :', z1 - z2)   # (1+4j)
print('z1 * z2  :', z1 * z2)   # (5+1j)
print('z1 / z2  :', z1 / z2)   # (-0.5+2.5j)

# Conjugate: flip sign of imaginary part
# MATLAB: conj(z1)
print('conj(z1) :', np.conj(z1))   # (2-3j)

# |z|^2 = z * conj(z) = a^2 + b^2
print('|z1|^2   :', z1 * np.conj(z1))   # (13+0j)


# 3. Polar Form and Euler's Formula

Every complex number can be written in polar form:

**z = r · e^(jθ)**

where r = |z| is the **magnitude** and θ = ∠z is the
**phase angle** (in radians).

**Euler's formula** links the two forms:

**e^(jθ) = cos(θ) + j·sin(θ)**

This is the single most important identity in DSP.
It means a complex exponential is a unit-magnitude
phasor rotating at angle θ on the complex plane.

| Quantity | Formula | NumPy | MATLAB |
|----------|---------|-------|--------|
| Magnitude | r = √(a²+b²) | `np.abs(z)` | `abs(z)` |
| Phase | θ = atan2(b,a) | `np.angle(z)` | `angle(z)` |
| Polar → rect | a+jb = r·e^(jθ) | `r*np.exp(1j*theta)` | `r*exp(1j*theta)` |


In [ ]:
z = 3 + 4j

# Magnitude — MATLAB: abs(z)
print('|z|      :', np.abs(z))         # 5.0

# Phase angle in radians — MATLAB: angle(z)
print('angle(z) :', np.angle(z).round(4))

# Phase in degrees — MATLAB: angle(z)*180/pi
print('angle(z) :', np.angle(z, deg=True).round(2), 'deg')

# Euler's formula: e^(j*theta)
# MATLAB: exp(1j * theta)
theta = np.pi / 4   # 45 degrees
phasor = np.exp(1j * theta)
print('e^(j*pi/4)      :', phasor.round(4))
print('cos+j*sin check :', 
      (np.cos(theta) + 1j*np.sin(theta)).round(4))
print('magnitude       :', np.abs(phasor))  # always 1.0

# Reconstruct z from polar form
r     = np.abs(z)
theta = np.angle(z)
z_reconstructed = r * np.exp(1j * theta)
print('reconstructed   :', z_reconstructed.round(6))


In [ ]:
# Visualise Euler's formula: e^(j*theta) for theta in [0, 2pi]
theta = np.linspace(0, 2*np.pi, 300)
circle = np.exp(1j * theta)   # unit circle

# Mark four key angles
angles  = [0, np.pi/2, np.pi, 3*np.pi/2]
labels  = ['1', 'j', '-1', '-j']
colours = ['C0','C1','C2','C3']

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(circle.real, circle.imag, 'k', lw=1)

for a, lbl, col in zip(angles, labels, colours):
    pt = np.exp(1j * a)
    ax.annotate(f'e^(jπ·{a/np.pi:.1g}) = {lbl}',
                xy=(pt.real, pt.imag),
                xytext=(pt.real*1.25, pt.imag*1.25),
                ha='center', color=col, fontsize=9)
    ax.plot(pt.real, pt.imag, 'o', color=col, ms=8)

ax.axhline(0, color='k', lw=0.5)
ax.axvline(0, color='k', lw=0.5)
ax.set(xlabel='Real', ylabel='Imaginary',
       title="Euler's formula — unit circle",
       aspect='equal', xlim=(-1.6,1.6), ylim=(-1.5,1.5))
plt.tight_layout()
plt.show()


# 4. NumPy Operations on Complex Arrays

NumPy handles complex arrays natively — all standard
functions work element-wise on complex dtype arrays.

| Operation | NumPy | MATLAB |
|-----------|-------|--------|
| Magnitude | `np.abs(z)` | `abs(z)` |
| Phase | `np.angle(z)` | `angle(z)` |
| Real part | `z.real` or `np.real(z)` | `real(z)` |
| Imag part | `z.imag` or `np.imag(z)` | `imag(z)` |
| Conjugate | `np.conj(z)` | `conj(z)` |
| Unit phasor | `np.exp(1j*theta)` | `exp(1j*theta)` |
| Complex dtype | `np.complex128` | `double` (complex) |


In [ ]:
# Array of complex channel coefficients
np.random.seed(0)
n = 8
# CN(0,1): complex Gaussian with unit variance
# Each component ~ N(0, 1/sqrt(2))
h = (np.random.randn(n) + 1j*np.random.randn(n))
h /= np.sqrt(2)   # normalise to unit mean power

print('dtype    :', h.dtype)   # complex128
print('shape    :', h.shape)

# Element-wise operations
print('|h|      :', np.abs(h).round(3))
print('angle(h) :', np.angle(h, deg=True).round(1))
print('h.real   :', h.real.round(3))
print('h.imag   :', h.imag.round(3))

# Mean power = E[|h|^2]
# MATLAB: mean(abs(h).^2)
print('mean power:', np.mean(np.abs(h)**2).round(3))

# Conjugate transpose (Hermitian) of a matrix
# MATLAB: H'
H = h.reshape(2, 4)
H_H = H.conj().T   # Hermitian transpose
print('H shape   :', H.shape, '-> H^H:', H_H.shape)


# 5. Argand (Phasor) Diagram

An Argand diagram plots complex numbers on a 2-D plane:
the real axis (x) and the imaginary axis (y).
In wireless, this is the **constellation diagram** or
**IQ plot** — every symbol is a point on this plane.


In [ ]:
# Plot a QPSK constellation (ideal + noisy)
np.random.seed(1)
n_sym = 200

# Ideal QPSK symbols: (±1 ± j) / sqrt(2)
ideal = np.array([1+1j, -1+1j, -1-1j, 1-1j]) / np.sqrt(2)

# Random symbol indices
idx = np.random.randint(0, 4, n_sym)

# Add complex Gaussian noise
noise_std = 0.15
noise = noise_std*(np.random.randn(n_sym)
                 + 1j*np.random.randn(n_sym))
received = ideal[idx] + noise

fig, ax = plt.subplots(figsize=(5, 4))

# Scatter plot of received symbols
ax.scatter(received.real, received.imag,
           s=8, alpha=0.5, label='Received')

# Mark ideal constellation points
ax.scatter(ideal.real, ideal.imag,
           s=120, marker='*', color='red',
           zorder=5, label='Ideal')

ax.axhline(0, color='k', lw=0.5)
ax.axvline(0, color='k', lw=0.5)
ax.set(xlabel='In-phase (I)', ylabel='Quadrature (Q)',
       title='QPSK Constellation — Argand Diagram',
       aspect='equal', xlim=(-1.2,1.2), ylim=(-1.2,1.2))
ax.legend()
plt.tight_layout()
plt.show()


# 6. Rotating Phasors and Complex Exponentials

The complex exponential **e^(j2πft)** represents a phasor
rotating at frequency f (Hz). As time progresses:
- Its real part traces `cos(2πft)`
- Its imaginary part traces `sin(2πft)`

This is the complex baseband representation of a
carrier wave — the foundation of IQ modulation.
Plotting I vs Q vs time gives a **helix**;
projecting onto I or Q gives the real sinusoid.


In [ ]:
f  = 3      # Hz
fs = 200    # samples/s
t  = np.linspace(0, 1, fs, endpoint=False)

# Complex exponential — the rotating phasor
z  = np.exp(1j * 2 * np.pi * f * t)

fig, axes = plt.subplots(1, 3, figsize=(5, 2.5))

# IQ plane: traces the unit circle
axes[0].plot(z.real, z.imag, lw=0.8)
axes[0].set(xlabel='I', ylabel='Q',
            title='IQ plane', aspect='equal')

# Real projection: cosine
axes[1].plot(t, z.real)
axes[1].set(xlabel='t (s)', ylabel='I',
            title='Real part = cos')

# Imaginary projection: sine
axes[2].plot(t, z.imag, color='C1')
axes[2].set(xlabel='t (s)', ylabel='Q',
            title='Imag part = sin')

plt.tight_layout()
plt.show()


In [ ]:
# Adding two complex exponentials = superposition
# MATLAB: z = exp(1j*2*pi*f1*t) + 0.5*exp(1j*2*pi*f2*t)
f1, f2 = 3, 7
z_sum = (np.exp(1j*2*np.pi*f1*t)
       + 0.5*np.exp(1j*2*np.pi*f2*t))

fig, axes = plt.subplots(1, 2, figsize=(5, 2.5))

axes[0].plot(z_sum.real, z_sum.imag, lw=0.6, alpha=0.8)
axes[0].set(xlabel='I', ylabel='Q',
            title='IQ — two phasors', aspect='equal')

axes[1].plot(t, z_sum.real, label='I')
axes[1].plot(t, z_sum.imag, label='Q')
axes[1].set(xlabel='t (s)', title='Time domain')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()


# 7. dB and dBm

Decibels appear everywhere in RF and DSP work.

| Quantity | Formula | Use case |
|----------|---------|----------|
| Power ratio (dB) | 10·log₁₀(P/P_ref) | SNR, path loss, filter gain |
| Amplitude ratio (dB) | 20·log₁₀(A/A_ref) | Voltage, field strength |
| Absolute power (dBm) | 10·log₁₀(P_mW) | TX power, received power |

The factor is **10 for power, 20 for amplitude** because
power ∝ amplitude², and log(x²) = 2·log(x).

MATLAB and Python use identical formulas:
`10*log10(x)` and `20*log10(x)`.


In [ ]:
# Power in dB — MATLAB: 10*log10(snr_linear)
snr_linear = np.array([1, 2, 10, 100, 1000])
snr_db     = 10 * np.log10(snr_linear)
print('SNR linear:', snr_linear)
print('SNR dB    :', snr_db)

# dB back to linear — MATLAB: 10.^(snr_db/10)
snr_back = 10 ** (snr_db / 10)
print('back to linear:', snr_back)

# Amplitude to dB — MATLAB: 20*log10(A)
amplitudes = np.array([0.001, 0.1, 1.0, 10.0])
amp_db = 20 * np.log10(amplitudes)
print('amplitude dB  :', amp_db)

# dBm: power relative to 1 mW
# P_dBm = 10*log10(P_watts * 1000)
p_watts = np.array([1e-6, 1e-3, 1.0])   # 1uW, 1mW, 1W
p_dbm   = 10 * np.log10(p_watts * 1000)
print('dBm           :', p_dbm)   # [-30, 0, 30]

# Quick reference:
#   +3 dB  ≈ double the power
#   -3 dB  ≈ half the power
#   +10 dB = 10x power
#   +20 dB = 100x power  (10x amplitude)
print('3 dB in linear:', 10**(3/10))   # ~2


# 8. MATLAB → Python Quick Reference

| Operation | MATLAB | Python / NumPy |
|-----------|--------|---------------|
| Complex literal | `1 + 2i` | `1 + 2j` |
| Imaginary unit | `1i` or `1j` | `1j` |
| Real part | `real(z)` | `z.real` or `np.real(z)` |
| Imaginary part | `imag(z)` | `z.imag` or `np.imag(z)` |
| Magnitude | `abs(z)` | `np.abs(z)` |
| Phase (rad) | `angle(z)` | `np.angle(z)` |
| Phase (deg) | `angle(z)*180/pi` | `np.angle(z, deg=True)` |
| Conjugate | `conj(z)` | `np.conj(z)` or `z.conj()` |
| Conj. transpose | `A'` | `A.conj().T` |
| Unit phasor | `exp(1j*theta)` | `np.exp(1j*theta)` |
| Euler's formula | `exp(1j*pi)` | `np.exp(1j*np.pi)` |
| Complex array | `complex(a,b)` | `a + 1j*b` |
| Real sinusoid | `cos(2*pi*f*t)` | `np.cos(2*np.pi*f*t)` |
| Power to dB | `10*log10(P)` | `10*np.log10(P)` |
| Amp to dB | `20*log10(A)` | `20*np.log10(A)` |
| dB to linear | `10.^(x/10)` | `10**(x/10)` |
| dBm | `10*log10(P*1e3)` | `10*np.log10(P*1e3)` |
